In [4]:
import numpy as np
import gvar as gv
import scipy
import functools
import matplotlib
import matplotlib.pyplot as plt
import h5py
import os 
import sys 
# local imports 
sys.path.append('../corrfit-exotics')
import corrfit

In [13]:
import operator_factory
from typing import List 
from operator_factory import QuantumNum
# operators = List[QuantumNum]
operators = operator_factory.a1_mp
for src_idx, (src_name, src_op) in enumerate(operators.items()):  # src_idx is an integer index
    for snk_idx, (snk_name, snk_op) in enumerate(operators.items()):
        print(src_idx, (src_name, src_op))
        print('===')
        print(snk_idx, (snk_name, snk_op))
channel= 'a1_mp'
print(f'operator_factory.{channel}')
for i, op in enumerate(operators):
    print(operators[op].strange)

0 ('pion', QuantumNum(name='pion', had=1, F='A1', twoI=1, strange=0, S=0, P=-1, C=1, gamma=array([[ 1.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  1.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j, -1.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j]]), gamma_i=False, deriv=None, mom='mom_0_0_0'))
===
0 ('pion', QuantumNum(name='pion', had=1, F='A1', twoI=1, strange=0, S=0, P=-1, C=1, gamma=array([[ 1.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  1.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j, -1.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j]]), gamma_i=False, deriv=None, mom='mom_0_0_0'))
0 ('pion', QuantumNum(name='pion', had=1, F='A1', twoI=1, strange=0, S=0, P=-1, C=1, gamma=array([[ 1.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  1.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j, -1.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j]]), gamma_i=False, deriv=None, mom='mom_0_0_0'))
===
1 ('pion_2', QuantumNum(name='pion_2',

In [11]:
'''
solve gevp on correlator matrix at times t0, td
''' 

# 1. read in raw correlators from h5 file 
h5_path = os.path.abspath('/home/grant/jureca_exolaunch')
a1_mp_file = os.path.join(h5_path,'contractions/a1_mp_nvec_96_tsrc_24_task1.h5')
data = np.zeros((5,2, 96, 24))
tsrc_avg = True
        
expected_cfgs = {f'cfg_{i}' for i in [11,12]}
file_h5 = a1_mp_file
with h5py.File(file_h5, 'r') as f:
    for iop,op in enumerate(f):
        for isrc, tsrc in enumerate(f[op]):
            t = int(tsrc.split('_')[1])
            cfg_list = list(f[f'/{op}/{tsrc}/'])
            for icfg, cfg in enumerate(sorted(cfg_list, key=lambda cfg: int(cfg[4:]))):
                if cfg in expected_cfgs:
                    cfg_data = np.array(f[f'/{op}/{tsrc}/{cfg}'][:])
                    data[iop,icfg, :, t] = cfg_data

    if tsrc_avg:
        for i in range(24):
            data[:,:, :, i] = np.roll(data[:,:, :, i], -4*i, axis=2)
        data = data.mean(axis=2)

data.shape
# vals, vecs = scipy.linalg.eigh(corr_array[td, :, :], corr_array[t0, :, :], eigvals_only=False)


/tmp/ipykernel_22710/648162496.py:21: ComplexWarning: Casting complex values to real discards the imaginary part
  data[iop,icfg, :, t] = cfg_data


(5, 2, 24)